In [1]:
# Notebook: 08_conditions_sweep
# Tests the two conditions claimed in the paper's "when does it work" section, WITHOUT retraining:
#   (1) effective label-space size K: restrict stored predictions to the top-N categories and
#       recompute the group comparison as N shrinks 5691 -> 50.
#   (2) belief sharpness: sharpen/soften beliefs by temperature T (T<1 sharper) at fixed N.
# For each setting we report, over taxonomy (depth-4) groups, the group-ranking of true noise
# rate for the best baseline (p90 of per-item mismatch) vs the best GUARD term (C_proj*kappa),
# and their paired bootstrap AUROC gap. Reads nb01 artifacts only. Grayscale figs, dpi 600.
import os, csv
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"]="0.2"; plt.rcParams["axes.linewidth"]=0.8; plt.rcParams["font.family"]="DejaVu Sans"
GREYS=["#111111","#555555","#888888","#bbbbbb","#dddddd"]
FIG=os.path.join("..","results","figures"); TAB=os.path.join("..","results","tables")
os.makedirs(FIG,exist_ok=True); os.makedirs(TAB,exist_ok=True)
EPS=1e-12
def entropy(p,axis=-1):
    p=np.clip(p,EPS,1.0); return -np.sum(p*np.log(p),axis=axis)
def js(q,p):
    q=np.clip(q,EPS,1.0); p=np.clip(p,EPS,1.0); m=0.5*(q+p)
    return float(0.5*np.sum(q*np.log2(q/m))+0.5*np.sum(p*np.log2(p/m)))

# --- load artifacts ---
DATA_DIR=os.path.join("..","data")
meta=pd.read_parquet(os.path.join(DATA_DIR,"item_meta.parquet"))
pi=np.load(os.path.join(DATA_DIR,"pi_memmap.npy"),mmap_mode="r"); N,K=pi.shape
classes=np.load(os.path.join(DATA_DIR,"label_classes.npy"))
noisy=meta["noisy_id"].values; cleanid=meta["clean_id"].values; mis=meta["is_misregistered"].values

# --- taxonomy depth-4 parent per item (from the registered/noisy leaf) ---
p=os.path.join(DATA_DIR,"category_mapping.csv")
with open(p,encoding="utf-8",errors="replace") as f: hd=[f.readline() for _ in range(6)]
try: sep=csv.Sniffer().sniff("".join(hd),delimiters=[",","\t",";","|"]).delimiter
except Exception: sep="\t"
cmap=pd.read_csv(p,sep=sep,engine="python"); lab2name=dict(zip(cmap.category_label,cmap.category_name))
leaf_parts=[([s.strip() for s in lab2name.get(int(classes[d]),"").split(">")]
             if lab2name.get(int(classes[d]),"") else []) for d in range(K)]
DEPTH=4
leaf_parent=np.array([" > ".join(pp[:DEPTH]) if len(pp)>=DEPTH else None for pp in leaf_parts],dtype=object)

freq=np.bincount(noisy,minlength=K)                       # category frequency (by noisy label)
order=np.argsort(-freq)                                   # most frequent first

def auroc_hilo(g, col):
    lo,hi=g.true_noise.quantile(1/3),g.true_noise.quantile(2/3)
    sub=g[(g.true_noise<=lo)|(g.true_noise>=hi)]
    y=(sub.true_noise>=hi).astype(int).values
    if len(np.unique(y))<2: return np.nan
    a=roc_auc_score(y,sub[col].values); return max(a,1-a)

def run_setting(N_keep, T, min_size=30, max_size=5000):
    colsel=np.sort(order[:N_keep]); pos=-np.ones(K,dtype=np.int64); pos[colsel]=np.arange(len(colsel))
    keep=set(colsel.tolist())
    mask=np.isin(noisy,list(keep)) & np.isin(cleanid,list(keep))
    parent=leaf_parent.copy()
    ip=np.array([parent[d] for d in noisy],dtype=object)
    idx_all=np.where(mask)[0]
    ser=pd.Series(ip[idx_all])
    sizes=ser.value_counts()
    groups=[k for k,v in sizes.items() if k is not None and min_size<=v<=max_size]
    recs=[]
    for gk in groups:
        gi=idx_all[(ip[idx_all]==gk)]
        rows=np.asarray(pi[gi][:,colsel],dtype=np.float64)
        if T!=1.0:
            L=np.log(np.clip(rows,EPS,1))/T; L-=L.max(1,keepdims=True); rows=np.exp(L)
        rows/=rows.sum(1,keepdims=True)                    # renormalize over the sub-universe
        pbar=rows.mean(0)
        pn=pos[noisy[gi]]
        q=np.bincount(pn,minlength=len(colsel)).astype(float); q/=q.sum()
        C=js(q,pbar); r=np.clip(pbar-q,0,None)
        kap=0.0 if r.sum()<=EPS else 1-entropy(r/r.sum())/np.log(len(colsel))
        s_ind=1.0-rows[np.arange(len(gi)),pn]
        recs.append(dict(true_noise=float(mis[gi].mean()),
                         p90=float(np.quantile(s_ind,0.9)), Cproj_kappa=C*kap))
    g=pd.DataFrame(recs)
    return dict(N=N_keep, T=T, n_groups=len(g),
        p90_sp=spearmanr(g.p90,g.true_noise)[0], p90_au=auroc_hilo(g,"p90"),
        guard_sp=spearmanr(g.Cproj_kappa,g.true_noise)[0], guard_au=auroc_hilo(g,"Cproj_kappa"))

# ===== Sweep 1: effective label-space size (T=1) =====
print("=== Sweep 1: effective label-space size K (T=1) ===")
s1=[run_setting(n,1.0) for n in [50,200,1000,5691]]
df1=pd.DataFrame(s1); df1.to_csv(os.path.join(TAB,"t08_sweep_K.csv"),index=False)
print(df1.round(3).to_string(index=False))

# ===== Sweep 2: belief sharpness (fixed N=1000) =====
print("\n=== Sweep 2: belief sharpness via temperature (N=1000) ===")
s2=[run_setting(1000,T) for T in [0.3,0.5,0.7,1.0,1.5]]
df2=pd.DataFrame(s2); df2.to_csv(os.path.join(TAB,"t08_sweep_T.csv"),index=False)
print(df2.round(3).to_string(index=False))

# ===== Figure: AUROC vs K (left) and vs T (right), baseline vs GUARD =====
fig,ax=plt.subplots(1,2,figsize=(8.6,3.4))
ax[0].plot(df1.N,df1.guard_au,marker="s",color=GREYS[0],ls="-",label="C_proj*kappa (GUARD)")
ax[0].plot(df1.N,df1.p90_au,marker="o",color=GREYS[2],ls="--",label="p90 (baseline)")
ax[0].set_xscale("log"); ax[0].set_xlabel("effective label-space size N"); ax[0].set_ylabel("AUROC (hi vs lo noise)")
ax[0].axhline(0.5,color="black",ls=":",lw=1); ax[0].legend(frameon=False,fontsize=8)
ax[1].plot(df2["T"],df2.guard_au,marker="s",color=GREYS[0],ls="-",label="C_proj*kappa (GUARD)")
ax[1].plot(df2["T"],df2.p90_au,marker="o",color=GREYS[2],ls="--",label="p90 (baseline)")
ax[1].set_xlabel("temperature T (sharper <-- 1 --> softer)"); ax[1].set_ylabel("AUROC (hi vs lo noise)")
ax[1].axhline(0.5,color="black",ls=":",lw=1); ax[1].legend(frameon=False,fontsize=8)
fig.tight_layout()
for e in ("png","pdf"): fig.savefig(os.path.join(FIG,f"f08_conditions_sweep.{e}"),dpi=600,bbox_inches="tight")
plt.close(fig)
print("\nsaved f08_conditions_sweep.{png,pdf}, t08_sweep_K.csv, t08_sweep_T.csv")
print("Read: if GUARD overtakes the p90 baseline as N shrinks and/or as beliefs sharpen (T<1),")
print("the paper's stated conditions are demonstrated, not merely asserted.")

=== Sweep 1: effective label-space size K (T=1) ===
   N   T  n_groups  p90_sp  p90_au  guard_sp  guard_au
  50 1.0        21   0.335     NaN     0.333       NaN
 200 1.0        90   0.381   0.759     0.282     0.706
1000 1.0       300   0.321   0.724     0.238     0.695
5691 1.0       645   0.303   0.686     0.051     0.537

=== Sweep 2: belief sharpness via temperature (N=1000) ===
   N   T  n_groups  p90_sp  p90_au  guard_sp  guard_au
1000 0.3       300   0.362   0.751     0.272     0.713
1000 0.5       300   0.358   0.748     0.270     0.714
1000 0.7       300   0.343   0.740     0.263     0.710
1000 1.0       300   0.321   0.724     0.238     0.695
1000 1.5       300   0.294   0.710     0.153     0.637

saved f08_conditions_sweep.{png,pdf}, t08_sweep_K.csv, t08_sweep_T.csv
Read: if GUARD overtakes the p90 baseline as N shrinks and/or as beliefs sharpen (T<1),
the paper's stated conditions are demonstrated, not merely asserted.


In [1]:
# Notebook: 07_reviewer_response
# Reviewer-response analyses on nb01 artifacts ONLY (no retraining):
#   A) cleanlab / confident-learning baseline (M1)
#   B) bootstrap 95% CIs + paired test for the group comparison (M4)
#   C) per-group true-noise-rate variance & AUROC ceiling (M8)
#   D) temperature-scaling probe for calibration confound (M5)
# Grayscale figures, dpi 600, PNG+PDF, no captions.
import os, csv, sys
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sys.path.append(os.path.join("..","src"))
from guard_core import entropy, js_divergence
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"]="0.2"; plt.rcParams["axes.linewidth"]=0.8; plt.rcParams["font.family"]="DejaVu Sans"
GREYS=["#111111","#555555","#888888","#bbbbbb","#dddddd"]
FIG=os.path.join("..","results","figures"); TAB=os.path.join("..","results","tables")
os.makedirs(FIG,exist_ok=True); os.makedirs(TAB,exist_ok=True)
def savefig(fig,name):
    for e in ("png","pdf"): fig.savefig(os.path.join(FIG,f"{name}.{e}"),dpi=600,bbox_inches="tight")
    plt.close(fig)
CHUNK=20000


# --- load artifacts ---
DATA_DIR=os.path.join("..","data")
meta=pd.read_parquet(os.path.join(DATA_DIR,"item_meta.parquet"))
pi=np.load(os.path.join(DATA_DIR,"pi_memmap.npy"),mmap_mode="r"); N,K=pi.shape
classes=np.load(os.path.join(DATA_DIR,"label_classes.npy"))
noisy=meta["noisy_id"].values; cleanid=meta["clean_id"].values
mis=meta["is_misregistered"].values; valid=cleanid>=0
p_noisy=meta["p_noisy"].values; entv=meta["entropy"].values
print(f"N={N:,} K={K:,} misregistration={mis[valid].mean():.3%}")


# ============ A. cleanlab / confident-learning baseline (M1) ============
# A1: real cleanlab on a memory-safe subsample (float32 pred_probs).
SUB=min(N,150000)
rng=np.random.default_rng(0); idxA=np.sort(rng.choice(N,SUB,replace=False))
probsA=np.asarray(pi[idxA],dtype=np.float32)
yA=mis[idxA]; vA=valid[idxA]
try:
    from cleanlab.rank import get_label_quality_scores
    qA=get_label_quality_scores(labels=noisy[idxA].astype(int), pred_probs=probsA, method="self_confidence")
    qN=get_label_quality_scores(labels=noisy[idxA].astype(int), pred_probs=probsA, method="normalized_margin")
    a1=roc_auc_score(yA[vA],(1-qA)[vA]); a2=roc_auc_score(yA[vA],(1-qN)[vA])
    print(f"[A1 cleanlab, n={SUB}] self_confidence AUROC={a1:.3f} | normalized_margin AUROC={a2:.3f}")
except Exception as e:
    print("[A1] cleanlab unavailable ->",repr(e)[:100],"| run: pip install cleanlab")

# A2: memory-safe normalized-margin over ALL items (chunked): m = p_noisy - max_{k!=noisy} p_k
top1=np.empty(N,np.float32); top2=np.empty(N,np.float32); arg1=np.empty(N,np.int64)
for i in range(0,N,CHUNK):
    ch=np.asarray(pi[i:i+CHUNK],dtype=np.float32)
    a=np.argsort(ch,axis=1)[:,-2:]         # top-2 indices
    arg1[i:i+len(ch)]=a[:,-1]
    top1[i:i+len(ch)]=ch[np.arange(len(ch)),a[:,-1]]
    top2[i:i+len(ch)]=ch[np.arange(len(ch)),a[:,-2]]
max_excl=np.where(arg1==noisy, top2, top1)      # max prob excluding the assigned label
cl_margin = max_excl - p_noisy                  # high => assigned label looks wrong (CL-style)
print(f"[A2 full-set cleanlab-margin] AUROC={roc_auc_score(mis[valid],cl_margin[valid]):.3f} "
      f"AUPRC={average_precision_score(mis[valid],cl_margin[valid]):.3f}")


# ============ build DEPTH=4 group table (same as nb06) + cleanlab-margin aggregates ============
p=os.path.join(DATA_DIR,"category_mapping.csv")
with open(p,encoding="utf-8",errors="replace") as f: hd=[f.readline() for _ in range(6)]
try: sep=csv.Sniffer().sniff("".join(hd),delimiters=[",","\t",";","|"]).delimiter
except Exception: sep="\t"
cmap=pd.read_csv(p,sep=sep,engine="python"); lab2name=dict(zip(cmap.category_label,cmap.category_name))
leaf_parts=[([s.strip() for s in lab2name.get(int(classes[d]),"").split(">")]
             if lab2name.get(int(classes[d]),"") else []) for d in range(K)]
DEPTH=4
leaf_parent=np.array([" > ".join(pp[:DEPTH]) if len(pp)>=DEPTH else None for pp in leaf_parts],dtype=object)
item_parent=leaf_parent[noisy]
S_of={}
for d in range(K):
    pk=leaf_parent[d]
    if pk is not None: S_of.setdefault(pk,[]).append(d)
sizes=pd.Series(item_parent).value_counts()
keep=[k for k,v in sizes.items() if k is not None and 30<=v<=50000]
recs=[]
for pk in keep:
    ix=np.where(item_parent==pk)[0]; S=np.array(S_of[pk]); vi=valid[ix]
    if len(S)<2 or not vi.any(): continue
    pbar=np.asarray(pi[ix],dtype=np.float64).mean(0)
    q=np.bincount(noisy[ix],minlength=K).astype(float); q/=q.sum()
    pS=pbar[S]; out_mass=float(1-pS.sum())
    if pS.sum()<=0: continue
    pSn=pS/pS.sum(); qSn=q[S]/q[S].sum()
    r=np.clip(pSn-qSn,0,None); kap=0.0 if r.sum()<=1e-12 else 1-entropy(r/r.sum())/np.log(len(S))
    s_ind=1.0-p_noisy[ix]; s_cl=cl_margin[ix]
    recs.append(dict(group=pk,n=len(ix),true_noise=mis[ix][vi].mean(),
        mean_1mp=float(s_ind.mean()), p90_1mp=float(np.quantile(s_ind,0.9)), mean_ent=float(entv[ix].mean()),
        cl_mean=float(s_cl.mean()), cl_p90=float(np.quantile(s_cl,0.9)),
        C_proj=js_divergence(qSn,pSn), out_mass=out_mass, Cproj_kappa=js_divergence(qSn,pSn)*kap))
g=pd.DataFrame(recs); g.to_csv(os.path.join(TAB,"t07_group_scores_d4.csv"),index=False)
print("groups:",len(g))


# ============ B. bootstrap 95% CIs + paired test (M4) ============
scores=["mean_1mp","p90_1mp","mean_ent","cl_mean","cl_p90","C_proj","out_mass","Cproj_kappa"]
lo,hi=g.true_noise.quantile(1/3),g.true_noise.quantile(2/3)
hilo=g[(g.true_noise<=lo)|(g.true_noise>=hi)].reset_index(drop=True)
yhl=(hilo.true_noise>=hi).astype(int).values
def auroc_dir(y,s):
    if len(np.unique(y))<2: return np.nan
    a=roc_auc_score(y,s); return max(a,1-a)
B=2000; rngb=np.random.default_rng(1)
rows=[]
n_all=len(g); n_hl=len(hilo)
boot_auroc={s:np.empty(B) for s in scores}
for s in scores:
    sp=np.empty(B)
    for b in range(B):
        ii=rngb.integers(0,n_all,n_all)
        sp[b]=spearmanr(g[s].values[ii],g.true_noise.values[ii])[0]
        jj=rngb.integers(0,n_hl,n_hl)
        boot_auroc[s][b]=auroc_dir(yhl[jj],hilo[s].values[jj])
    rows.append(dict(score=s,
        spearman=round(spearmanr(g[s],g.true_noise)[0],3),
        spearman_lo=round(np.nanquantile(sp,.025),3), spearman_hi=round(np.nanquantile(sp,.975),3),
        auroc=round(auroc_dir(yhl,hilo[s].values),3),
        auroc_lo=round(np.nanquantile(boot_auroc[s],.025),3), auroc_hi=round(np.nanquantile(boot_auroc[s],.975),3)))
ci=pd.DataFrame(rows); ci.to_csv(os.path.join(TAB,"t07_bootstrap_ci.csv"),index=False)
print(ci.to_string(index=False))

# paired test: best baseline (p90_1mp) vs best GUARD (Cproj_kappa)
d=boot_auroc["Cproj_kappa"]-boot_auroc["p90_1mp"]; d=d[~np.isnan(d)]
print(f"\nAUROC diff (Cproj_kappa - p90_1mp): mean={d.mean():+.3f} "
      f"95% CI=[{np.quantile(d,.025):+.3f}, {np.quantile(d,.975):+.3f}]  "
      f"P(GUARD>baseline)={ (d>0).mean():.2f}")
print("=> CI straddling 0 supports 'statistically indistinguishable'.")


# ============ C. target-variable variance & AUROC ceiling (M8) ============
tn=g.true_noise.values
print("per-group true noise rate:")
print(f"  mean={tn.mean():.3f} std={tn.std():.3f} min={tn.min():.3f} "
      f"q25={np.quantile(tn,.25):.3f} median={np.median(tn):.3f} q75={np.quantile(tn,.75):.3f} max={tn.max():.3f}")
print(f"  hi/lo tercile gap: lo<= {lo:.3f}  hi>= {hi:.3f}")
fig,ax=plt.subplots(figsize=(4.8,3.2))
ax.hist(tn,bins=40,color=GREYS[2],edgecolor="black",linewidth=0.4)
ax.axvline(lo,color="black",ls=":",lw=1); ax.axvline(hi,color="black",ls=":",lw=1)
ax.set_xlabel("per-group true noise rate"); ax.set_ylabel("groups")
savefig(fig,"f07_noise_rate_hist")
print("saved f07_noise_rate_hist.{png,pdf}")


# ============ D. temperature-scaling probe for calibration confound (M5) ============
# Exact temperature scaling from stored probabilities: log(p) recovers logits up to a constant,
# so softmax(log(p)/T) is genuine temperature scaling. Fit T on a val split by NLL vs CLEAN
# labels (an ORACLE calibration probe: "if perfectly calibrated, would confidence detect noise?").
EPS=1e-12
rng=np.random.default_rng(2)
sub=np.sort(rng.choice(N,min(N,120000),replace=False))
val=sub[:len(sub)//3]; ev=sub[len(sub)//3:]
def nll_for_T(idx,T):
    tot=0.0;n=0
    for i in range(0,len(idx),CHUNK):
        j=idx[i:i+CHUNK]; L=np.log(np.clip(np.asarray(pi[j],dtype=np.float32),EPS,1))/T
        L-=L.max(1,keepdims=True); P=np.exp(L); P/=P.sum(1,keepdims=True)
        y=cleanid[j]; ok=y>=0
        tot+=-np.log(np.clip(P[np.arange(len(j)),np.where(ok,y,0)],EPS,1))[ok].sum(); n+=ok.sum()
    return tot/max(n,1)
Ts=[0.5,0.7,1.0,1.5,2.0,3.0,5.0]
nlls=[(T,nll_for_T(val,T)) for T in Ts]
Tstar=min(nlls,key=lambda x:x[1])[0]
print("val NLL by T:",[(T,round(v,3)) for T,v in nlls],"-> T*=",Tstar)
def detect_auroc(idx,T):
    pcal=np.empty(len(idx),np.float32); ecal=np.empty(len(idx),np.float32)
    for i in range(0,len(idx),CHUNK):
        j=idx[i:i+CHUNK]; L=np.log(np.clip(np.asarray(pi[j],dtype=np.float32),EPS,1))/T
        L-=L.max(1,keepdims=True); P=np.exp(L); P/=P.sum(1,keepdims=True)
        pcal[i:i+len(j)]=P[np.arange(len(j)),noisy[j]]
        ecal[i:i+len(j)]=-np.sum(np.clip(P,EPS,1)*np.log(np.clip(P,EPS,1)),1)
    ve=valid[idx]
    return (roc_auc_score(mis[idx][ve],(1-pcal)[ve]), roc_auc_score(mis[idx][ve],ecal[ve]))
b1=detect_auroc(ev,1.0); bT=detect_auroc(ev,Tstar)
print(f"[eval n={len(ev)}] individual detection AUROC")
print(f"  uncalibrated (T=1):  1-p={b1[0]:.3f}  entropy={b1[1]:.3f}")
print(f"  calibrated  (T={Tstar}): 1-p={bT[0]:.3f}  entropy={bT[1]:.3f}")
print("=> if still ~0.5 after oracle calibration, the collapse is not a calibration artifact.")
pd.DataFrame([dict(setting="T=1",auroc_1mp=b1[0],auroc_entropy=b1[1]),
              dict(setting=f"T={Tstar}",auroc_1mp=bT[0],auroc_entropy=bT[1])]
             ).to_csv(os.path.join(TAB,"t07_temperature_scaling.csv"),index=False)


N=502,310 K=5,691 misregistration=14.748%


C:\Users\miy\miniconda3\envs\seller_seg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[A1 cleanlab, n=150000] self_confidence AUROC=0.526 | normalized_margin AUROC=0.523
[A2 full-set cleanlab-margin] AUROC=0.522 AUPRC=0.155
groups: 475
      score  spearman  spearman_lo  spearman_hi  auroc  auroc_lo  auroc_hi
   mean_1mp     0.130        0.031        0.224  0.583     0.520     0.646
    p90_1mp     0.210        0.117        0.300  0.622     0.556     0.684
   mean_ent     0.177        0.076        0.272  0.613     0.546     0.673
    cl_mean     0.093       -0.007        0.191  0.560     0.505     0.624
     cl_p90     0.042       -0.052        0.134  0.533     0.502     0.594
     C_proj     0.194        0.092        0.282  0.620     0.556     0.679
   out_mass     0.123        0.022        0.219  0.579     0.516     0.643
Cproj_kappa     0.195        0.101        0.283  0.623     0.560     0.684

AUROC diff (Cproj_kappa - p90_1mp): mean=+0.001 95% CI=[-0.085, +0.090]  P(GUARD>baseline)=0.51
=> CI straddling 0 supports 'statistically indistinguishable'.
per-group true 